# Play Markov Stag Hunt

The window prompts for `player_0`, then each remaining agent. Press one movement key per agent; the final agent's keypress immediately executes the complete parallel step.

| Keys | Action |
|---|---|
| `A` or Left Arrow | move left |
| `D` or Right Arrow | move right |
| `W` or Up Arrow | move up |
| `S` or Down Arrow | move down |
| Space | stay |

Press **Backspace** to reset the round and environment, or **Escape** to quit.

In [1]:
from masa.envs.multiagent.tabular import MarkovStagHunt

SEED = 0
WINDOW_SIZE = 720


In [2]:
def play(seed=SEED):
    import pygame

    key_to_action = {
        pygame.K_a: 0, pygame.K_LEFT: 0,
        pygame.K_d: 1, pygame.K_RIGHT: 1,
        pygame.K_w: 2, pygame.K_UP: 2,
        pygame.K_s: 3, pygame.K_DOWN: 3,
        pygame.K_SPACE: 4,
    }
    action_names = {0: "left", 1: "right", 2: "up", 3: "down", 4: "stay"}
    env = MarkovStagHunt(
        render_mode="human",
        render_window_size=WINDOW_SIZE,
        max_moves=100,
    )
    observations, infos = env.reset(seed=seed)
    pending = {}
    round_number = 1
    clock = pygame.time.Clock()

    def announce_next():
        agent = env.agents[len(pending)]
        pygame.display.set_caption(f"MASA - Markov Stag Hunt | Round {round_number} | Input: {agent}")
        print(f"Round {round_number}: enter movement for {agent}")

    announce_next()
    try:
        running = True
        while running and not env.human_window_closed:
            for event in pygame.event.get():
                if not env.handle_pygame_event(event):
                    running = False
                    break
                if event.type != pygame.KEYDOWN:
                    continue
                if event.key == pygame.K_ESCAPE:
                    running = False
                    break
                if event.key == pygame.K_BACKSPACE:
                    observations, infos = env.reset(seed=seed)
                    pending.clear()
                    round_number = 1
                    print("Environment reset.")
                    announce_next()
                    continue
                if event.key not in key_to_action:
                    continue

                agent = env.agents[len(pending)]
                action = key_to_action[event.key]
                pending[agent] = action
                print(f"  {agent}: {action_names[action]}")
                if len(pending) < len(env.agents):
                    announce_next()
                    continue

                actions = dict(pending)
                pending.clear()
                observations, rewards, terminations, truncations, infos = env.step(actions)
                print("  simultaneous step -> rewards:", rewards)
                print("  results:", next(iter(infos.values()))["results"])
                unsafe = [agent for agent, obs in observations.items() if env.cost_fn(env.label_fn(obs))]
                if unsafe:
                    print("  mauled agents:", unsafe)
                if any(terminations.values()) or any(truncations.values()):
                    print("Episode finished; resetting.")
                    observations, infos = env.reset()
                    round_number = 1
                else:
                    round_number += 1
                announce_next()

            if running:
                env.render()
                clock.tick(30)
    finally:
        env.close()

play()


Round 1: enter movement for player_0
